In [5]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64

JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#### FIX ME #####
# Import the CRUD Python module and AnimalShelter class
from CRUD_Python_Module import AnimalShelter


###########################
# Data Manipulation / Model
###########################

# MongoDB credentials
username = "aacuser"
password = "Panophobia1"

# Connect to database through the CRUD module
db = AnimalShelter(username, password)

# Retrieve all animal records from MongoDB
df = pd.DataFrame.from_records(db.read({}))

# Remove MongoDB ObjectId because Dash cannot display it
if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)


#########################
# Dashboard Layout / View
#########################

app = JupyterDash(__name__)


# Load the Grazioso Salvare logo
image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())


app.layout = html.Div([

    # Grazioso Salvare logo with link to SNHU
    html.Center(
        html.A(
            href='https://www.snhu.edu',
            children=html.Img(
                src='data:image/png;base64,{}'.format(encoded_image.decode()),
                style={'width': '300px'}
            )
        )
    ),

    # Unique identifier
    html.Center(
        html.H1('Grazioso Salvare Dashboard - Blake Buhr')
    ),

    html.Hr(),

    # Interactive filtering controls
    html.Div([
        dcc.RadioItems(
            id='filter-type',
            options=[
                {
                    'label': 'Water Rescue',
                    'value': 'water'
                },
                {
                    'label': 'Mountain/Wilderness Rescue',
                    'value': 'mountain'
                },
                {
                    'label': 'Disaster/Individual Tracking',
                    'value': 'disaster'
                },
                {
                    'label': 'Reset',
                    'value': 'reset'
                }
            ],
            value='reset',
            labelStyle={
                'display': 'inline-block',
                'margin-right': '20px'
            }
        )
    ]),

    html.Hr(),

    # Interactive data table
    dash_table.DataTable(
        id='datatable-id',

        columns=[
            {
                "name": i,
                "id": i,
                "deletable": False,
                "selectable": True
            }
            for i in df.columns
        ],

        data=df.to_dict('records'),

        # User-friendly table features
        page_size=10,
        sort_action='native',
        filter_action='native',

        # Required for map selection
        row_selectable='single',
        selected_rows=[0]
    ),

    html.Br(),
    html.Hr(),

    # Secondary chart and geolocation chart
    html.Div(
        className='row',
        style={'display': 'flex'},

        children=[

            html.Div(
                id='graph-id',
                className='col s12 m6',
                style={'width': '50%'}
            ),

            html.Div(
                id='map-id',
                className='col s12 m6',
                style={'width': '50%'}
            )
        ]
    )
])


#############################################
# Interaction Between Components / Controller
#############################################


# -------------------------------------------------
# Filter the data table based on the selected rescue
# type.
# -------------------------------------------------

@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):

    # Water Rescue
    if filter_type == 'water':

        query = {
            "animal_type": "Dog",

            "breed": {
                "$in": [
                    "Labrador Retriever Mix",
                    "Chesapeake Bay Retriever",
                    "Newfoundland"
                ]
            },

            "sex_upon_outcome": "Intact Female",

            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        }

    # Mountain or Wilderness Rescue
    elif filter_type == 'mountain':

        query = {
            "animal_type": "Dog",

            "breed": {
                "$in": [
                    "German Shepherd",
                    "Alaskan Malamute",
                    "Old English Sheepdog",
                    "Siberian Husky",
                    "Rottweiler"
                ]
            },

            "sex_upon_outcome": "Intact Male",

            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        }

    # Disaster or Individual Tracking
    elif filter_type == 'disaster':

        query = {
            "animal_type": "Dog",

            "breed": {
                "$in": [
                    "Doberman Pinscher",
                    "German Shepherd",
                    "Golden Retriever",
                    "Bloodhound",
                    "Rottweiler"
                ]
            },

            "sex_upon_outcome": "Intact Male",

            "age_upon_outcome_in_weeks": {
                "$gte": 20,
                "$lte": 300
            }
        }

    # Reset to the complete unfiltered dataset
    else:
        query = {}

    # Retrieve the filtered records through the CRUD module
    filtered_data = pd.DataFrame.from_records(
        db.read(query)
    )

    # Remove MongoDB ObjectId if present
    if '_id' in filtered_data.columns:
        filtered_data.drop(columns=['_id'], inplace=True)

    return filtered_data.to_dict('records')


# -------------------------------------------------
# Create a secondary chart showing the breeds
# represented in the current data table.
# -------------------------------------------------

@app.callback(
    Output('graph-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data')]
)
def update_graphs(viewData):

    # Convert the current table data into a DataFrame
    if viewData is None or len(viewData) == 0:
        return html.Div("No data available for this selection.")

    dff = pd.DataFrame.from_dict(viewData)

    # Count the number of animals represented by each breed
    breed_counts = (
        dff['breed']
        .value_counts()
        .reset_index()
    )

    breed_counts.columns = ['breed', 'count']

    # Create a pie chart of the breeds
    figure = px.pie(
        breed_counts,
        names='breed',
        values='count',
        title='Animal Breeds in Current Selection'
    )

    return [
        dcc.Graph(
            figure=figure
        )
    ]


# -------------------------------------------------
# Highlight selected columns in the data table.
# -------------------------------------------------

@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):

    if selected_columns is None:
        return []

    return [
        {
            'if': {
                'column_id': i
            },

            'background_color': '#D2F3FF'
        }

        for i in selected_columns
    ]


# -------------------------------------------------
# Update the geolocation chart based on the selected
# row in the data table.
# -------------------------------------------------

@app.callback(
    Output('map-id', 'children'),

    [
        Input(
            'datatable-id',
            'derived_virtual_data'
        ),

        Input(
            'datatable-id',
            'derived_virtual_selected_rows'
        )
    ]
)
def update_map(viewData, index):

    # Prevent errors when the table has no data
    if viewData is None or len(viewData) == 0:
        return html.Div("No animal data available for mapping.")

    # Convert table data into a DataFrame
    dff = pd.DataFrame.from_dict(viewData)

    # If no row is selected, use the first row
    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]

    # Make sure the selected row still exists
    if row >= len(dff):
        row = 0

    # Austin, Texas coordinates
    return [
        dl.Map(
            style={
                'width': '100%',
                'height': '500px'
            },

            center=[
                30.75,
                -97.48
            ],

            zoom=10,

            children=[

                dl.TileLayer(
                    id="base-layer-id"
                ),

                # Marker showing the selected animal
                dl.Marker(

                    position=[
                        dff.iloc[row]['location_lat'],
                        dff.iloc[row]['location_long']
                    ],

                    children=[

                        dl.Tooltip(
                            dff.iloc[row]['breed']
                        ),

                        dl.Popup([

                            html.H1("Animal Name"),

                            html.P(
                                dff.iloc[row]['name']
                            )
                        ])
                    ]
                )
            ]
        )
    ]


# -------------------------------------------------
# Run the dashboard
# -------------------------------------------------

app.run_server()

Dash app running on https://cactusponcho-aliasclient-3000.codio.io/proxy/8050/
